![image.png](https://i.imgur.com/a3uAqnb.png)

# Text Summarization with Transformers

## 🎯 Objective
Fine-tune a pre-trained Seq2Seq Transformer model (**T5-small**) on the **SAMSum** dataset to summarize dialogues/conversations.

## 📚 What You'll Learn
- Encoder-Decoder (Seq2Seq) architecture vs Encoder-only
- Working with generation tasks
- Using ROUGE metrics for summarization
- The concept of "task prefixes" in T5 (e.g., "summarize: ")

---

## Step 1: Install required libraries

In [1]:
!pip install transformers datasets evaluate accelerate rouge_score sentencepiece -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.6 MB/s eta 0:00:00


## Step 2: Import libraries

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    pipeline
)
import evaluate

print("GPU available:", torch.cuda.is_available())

GPU available: False


## Step 3: Load the SAMSum dataset

SAMSum contains ~16,000 messenger-like dialogues with human-written summaries.

In [4]:
dataset = load_dataset("knkarthick/samsum")


print(dataset)

# View an example
example = dataset["train"][0]
print("\n=== DIALOGUE ===")
print(example["dialogue"])
print("\n=== SUMMARY ===")
print(example["summary"])

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

=== DIALOGUE ===
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

=== SUMMARY ===
Amanda baked cookies and will bring Jerry some tomorrow.


## Step 4: Load tokenizer and prepare data

T5 expects task prefixes. For summarization we use `"summarize: "`.

In [5]:
model_checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

prefix = "summarize: "
max_input_length = ...
max_target_length = ...

def preprocess(examples):
    inputs = [prefix + dialogue for dialogue in examples["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # Tokenize targets
    labels = tokenizer(text_target=examples["summary"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Use subset for fast training
train_dataset =
eval_dataset =
test_dataset =
train_tokenized=
eval_tokenized =

print("Tokenization done ✅")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenization done ✅


## Step 5: Load the pre-trained model

In [6]:
model = ...
data_collator = DataCollatorForSeq2Seq(tokenizer= ..., model=...)

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Step 6: Define ROUGE evaluation metric

**ROUGE** = Recall-Oriented Understudy for Gisting Evaluation. Measures how much overlap exists between generated and reference summaries.

In [7]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 with pad token in labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in result.items()}

## Step 7: Train the model

In [ ]:
training_args = ...

trainer = ...

trainer.train()

## Step 8: Evaluate

In [ ]:
results = trainer.evaluate()
print("\nFinal ROUGE scores:")
for key, value in results.items():
    if "rouge" in key.lower():
        print(f"  {key}: {value}")

## Step 9: Generate summaries using the fine-tuned model

In [ ]:
summarizer = pipeline(..., model= ..., tokenizer=...)

# Test on dialogues from the test set
for i in range(3):
    dialogue = test_dataset[i]["dialogue"]
    reference = test_dataset[i]["summary"]

    generated = summarizer(
        prefix + dialogue,
        max_length=max_target_length,
        min_length=10,
        do_sample=False
    )[0]["summary_text"]

    print(f"=== Example {i+1} ===")
    print("DIALOGUE:")
    print(dialogue)
    print(f"\n🟢 REFERENCE SUMMARY: {reference}")
    print(f"🔵 GENERATED SUMMARY: {generated}")
    print("\n" + "="*70 + "\n")

## Step 10: Try your own dialogue!

In [ ]:
my_dialogue = """
Ahmed: Hey, are we still meeting for lunch tomorrow?
Sara: Yes! Let's go to that new Italian place downtown.
Ahmed: Great. What time works for you?
Sara: Around 1 PM? I have a meeting until 12:30.
Ahmed: Perfect. I'll book a table for two.
Sara: Awesome, see you then!
"""

summary = summarizer(
    prefix + my_dialogue,
    max_length=50,
    min_length=10,
    do_sample=False
)[0]["summary_text"]

print("📝 Generated summary:")
print(summary)

## 📝 For further Exploration

1. **Try different decoding strategies**: change `do_sample=True`, add `num_beams=4`, and compare results.
2. **Try a different model** like `facebook/bart-base` and compare ROUGE scores.
3. **Use a different dataset** like `cnn_dailymail` (news summarization) and observe differences.
4. **Adjust the maximum summary length** and observe how it affects quality.
5. **Build your own input**: write a paragraph (Wikipedia article, news, etc.) and summarize it.

## Concepts to Discuss
- Why do we use a Seq2Seq model instead of a regular Transformer?
- What does the `summarize: ` prefix do in T5?
- What's the difference between ROUGE-1, ROUGE-2, and ROUGE-L?
- Why is summarization harder to evaluate than classification?